In [1]:
# =============================================================================
# NOTEBOOK 02 — DATA CLEANING & TRANSFORMATION
# =============================================================================
# Business Objective:
#   Transform raw OLA Electric Play Store reviews into a structured,
#   analysis-ready dataset that supports sentiment analysis, complaint
#   classification, temporal analysis, and statistical testing.
#
# Business Questions This Notebook Answers:
#   1. How much of the raw data is usable without significant loss?
#   2. What proportion of reviews are non-English and how does that
#      affect downstream VADER reliability?
#   3. What does the average rating trend look like across 2022-2026?
#
# Deliverables:
#   - data/cleaned/ola_reviews_cleaned.csv
#   - data/cleaned/ola_monthly_trends.csv
#   - outputs/observations.md (initialized and first entries logged)
# =============================================================================

In [2]:
import pandas as pd
import numpy as np
import re
import os
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded.")

Libraries loaded.


In [3]:
# Load raw dataset
df = pd.read_csv(r'C:\Users\Naren karthikeyan\OLA Analysis\data\raw\rawola_reviews_raw.csv')

# Capture original shape for validation report
original_shape = df.shape

print(f"Raw dataset loaded.")
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nSample row:")
df.head(2)

Raw dataset loaded.
Shape: (7119, 12)
Columns: ['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion', 'year_month']

Sample row:


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,year_month
0,9d7cc322-4e81-421f-8ea5-def70d881230,Sonu Kumar,https://play-lh.googleusercontent.com/a/ACg8oc...,I am facing a issue where I cannot open my ola...,1,0,5.3.0,2026-06-25 00:09:18,NaN,NaN,5.3.0,2026-06
1,240988a8-cf07-4259-9b6d-9fa7b22a433d,Ron,https://play-lh.googleusercontent.com/a/ACg8oc...,if there was no star I would have given that. ...,1,2,5.3.0,2026-06-24 22:00:18,NaN,NaN,5.3.0,2026-06


In [5]:
print("-" * 55)
print("PRE-CLEANING VALIDATION REPORT")
print("-" * 55)

issues_found = []

# CHECK 1 — Rating range (must be 1-5 only)
invalid_ratings = df[~df['score'].between(1, 5)]
if len(invalid_ratings) > 0:
    issues_found.append(f"CRITICAL: {len(invalid_ratings)} reviews with invalid ratings found.")
else:
    print("✓ Rating range check passed — all scores between 1 and 5")

# CHECK 2 — Empty review content
empty_reviews = df[df['content'].isna() | (df['content'].str.strip() == '')]
if len(empty_reviews) > 0:
    issues_found.append(f"CRITICAL: {len(empty_reviews)} empty reviews found.")
else:
    print("✓ Empty review check passed — no empty content fields")

# CHECK 3 — Duplicate review IDs
duplicate_ids = df.duplicated(subset='reviewId', keep=False).sum()
if duplicate_ids > 0:
    issues_found.append(f"WARNING: {duplicate_ids} duplicate review IDs found.")
else:
    print("✓ Duplicate ID check passed — all review IDs unique")

# CHECK 4 — Future dates
df['at'] = pd.to_datetime(df['at'])
future_dates = df[df['at'] > pd.Timestamp.now()]
if len(future_dates) > 0:
    issues_found.append(f"WARNING: {len(future_dates)} reviews with future timestamps.")
else:
    print("✓ Future date check passed — all timestamps valid")

# Summary
print("-" * 55)
if issues_found:
    for issue in issues_found:
        print(issue)
else:
    print("All checks passed. Safe to proceed with cleaning.")
print("-" * 55)

-------------------------------------------------------
PRE-CLEANING VALIDATION REPORT
-------------------------------------------------------
✓ Rating range check passed — all scores between 1 and 5
✓ Empty review check passed — no empty content fields
✓ Duplicate ID check passed — all review IDs unique
✓ Future date check passed — all timestamps valid
-------------------------------------------------------
All checks passed. Safe to proceed with cleaning.
-------------------------------------------------------


In [6]:
# Columns with zero analytical value for this project
# replyContent and repliedAt — 100% null, confirmed Phase 1
# userImage — URL string, no analytical value
# Note: appVersion KEPT per project decision — storage cheap, re-scraping expensive

columns_to_drop = ['replyContent', 'repliedAt', 'userImage']

# Engineer developer_replied BEFORE dropping replyContent
df['developer_replied'] = False  # Confirmed 0% reply rate — Phase 1 finding

# Now drop
df = df.drop(columns=columns_to_drop)

print(f"Columns dropped: {columns_to_drop}")
print(f"Shape after drop: {df.shape}")
print(f"Remaining columns: {df.columns.tolist()}")

Columns dropped: ['replyContent', 'repliedAt', 'userImage']
Shape after drop: (7119, 10)
Remaining columns: ['reviewId', 'userName', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'appVersion', 'year_month', 'developer_replied']


In [7]:
# Datetime already converted in Cell 3 validation
# Extract all temporal features now

df['year']       = df['at'].dt.year
df['month']      = df['at'].dt.month
df['year_month'] = df['at'].dt.to_period('M')
df['quarter']    = df['at'].dt.to_period('Q')

print("Temporal features engineered:")
print(f"  year        — {df['year'].nunique()} unique years: {sorted(df['year'].unique())}")
print(f"  month       — values 1-12")
print(f"  year_month  — {df['year_month'].nunique()} unique months")
print(f"  quarter     — {df['quarter'].nunique()} unique quarters")

Temporal features engineered:
  year        — 5 unique years: [np.int32(2022), np.int32(2023), np.int32(2024), np.int32(2025), np.int32(2026)]
  month       — values 1-12
  year_month  — 50 unique months
  quarter     — 17 unique quarters


In [8]:
# Business-friendly grouping for downstream analysis
# Negative = 1-2 star, Neutral = 3 star, Positive = 4-5 star

def assign_rating_category(score):
    if score <= 2:
        return 'Negative'
    elif score == 3:
        return 'Neutral'
    else:
        return 'Positive'

df['rating_category'] = df['score'].apply(assign_rating_category)

print("Rating category distribution:")
print(df['rating_category'].value_counts())
print(f"\nAs percentage:")
print((df['rating_category'].value_counts() / len(df) * 100).round(1))

Rating category distribution:
rating_category
Negative    4689
Positive    2054
Neutral      376
Name: count, dtype: int64

As percentage:
rating_category
Negative    65.9
Positive    28.9
Neutral      5.3
Name: count, dtype: float64


In [9]:
# Capture length BEFORE cleaning — per project decision
# Allows measuring how much preprocessing altered the text

df['raw_review_length'] = df['content'].str.len()
df['raw_word_count']    = df['content'].str.split().str.len()

print("Raw review length captured.")
print(f"\nRaw review length stats:")
print(df['raw_review_length'].describe().round(1))
print(f"\nRaw word count stats:")
print(df['raw_word_count'].describe().round(1))

Raw review length captured.

Raw review length stats:
count    7119.0
mean      117.3
std       128.7
min         1.0
25%        16.0
50%        72.0
75%       174.0
max       500.0
Name: raw_review_length, dtype: float64

Raw word count stats:
count    7119.0
mean       21.2
std        23.3
min         1.0
25%         3.0
50%        13.0
75%        31.0
max       147.0
Name: raw_word_count, dtype: float64


In [10]:
def clean_review_text(text):
    """
    Cleans review text for NLP analysis.
    
    Deliberately KEEPS:
    - Punctuation like ! and ? (VADER uses these as sentiment signals)
    - Negations like "not", "no", "never" (critical for sentiment accuracy)
    - Mixed case (VADER treats ALL CAPS as emphasis — intentional signal)
    
    Deliberately REMOVES:
    - URLs and email addresses (no sentiment value)
    - Repeated characters beyond 3 (e.g. "baaaaaad" → "baad")
    - Leading/trailing whitespace
    - Excess internal whitespace
    - Standalone numbers with no context
    """
    if not isinstance(text, str):
        return ""
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove email addresses
    text = re.sub(r'\S+@\S+', '', text)
    
    # Normalize repeated characters (keeping max 2)
    # "baaaad" → "baad", "!!!!!!" → "!!"
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    
    # Remove standalone numbers (e.g. "123" alone, but keep "3 months")
    text = re.sub(r'\b\d+\b(?!\s*(month|day|week|year|km|star|rating))', '', text)
    
    # Collapse multiple whitespace into single space
    text = re.sub(r'\s+', ' ', text)
    
    # Strip leading and trailing whitespace
    text = text.strip()
    
    return text

# Apply to content column
df['content_clean'] = df['content'].apply(clean_review_text)

print("Text cleaning applied.")
print(f"\nSample — before vs after cleaning:")
print("-" * 55)
for i in range(3):
    print(f"BEFORE: {df['content'].iloc[i]}")
    print(f"AFTER:  {df['content_clean'].iloc[i]}")
    print()

Text cleaning applied.

Sample — before vs after cleaning:
-------------------------------------------------------
BEFORE: I am facing a issue where I cannot open my ola electric app it stuck on loading phase than white blank screen. I get my new ola scooter on 6/06/2026, but after than I won't able to open app. I have to use it without the app. That's why I face so many problems.
AFTER:  I am facing a issue where I cannot open my ola electric app it stuck on loading phase than white blank screen. I get my new ola scooter on //, but after than I won't able to open app. I have to use it without the app. That's why I face so many problems.

BEFORE: if there was no star I would have given that. so many mental harrasment and issues with services and they could ruke out any service as per their convenience. so I recommend never buy anything from this company. and don't ever believe in Indian company especially in electronics or vehicles . one of my biggest mistakes regret it and now I have 

In [11]:
# Capture length AFTER cleaning
df['clean_review_length'] = df['content_clean'].str.len()
df['word_count']          = df['content_clean'].str.split().str.len()

# How much did cleaning change the text?
df['length_change'] = df['raw_review_length'] - df['clean_review_length']

print("Clean review length captured.")
print(f"\nClean review length stats:")
print(df['clean_review_length'].describe().round(1))

print(f"\nAverage characters removed by cleaning: {df['length_change'].mean():.1f}")
print(f"Max characters removed: {df['length_change'].max()}")

Clean review length captured.

Clean review length stats:
count    7119.0
mean      116.5
std       127.7
min         1.0
25%        16.0
50%        71.0
75%       173.0
max       500.0
Name: clean_review_length, dtype: float64

Average characters removed by cleaning: 0.8
Max characters removed: 105


In [12]:
# Short reviews = under 4 words
# Flagged, NOT deleted — they are valid sentiment signals
# VADER can score "worst scooter" just as validly as a paragraph

SHORT_REVIEW_THRESHOLD = 4

df['is_short_review'] = df['word_count'] < SHORT_REVIEW_THRESHOLD

print(f"Short review threshold: under {SHORT_REVIEW_THRESHOLD} words")
print(f"Short reviews flagged: {df['is_short_review'].sum()} ({df['is_short_review'].mean()*100:.1f}%)")
print(f"\nSample short reviews:")
print(df[df['is_short_review']]['content_clean'].head(10).tolist())

Short review threshold: under 4 words
Short reviews flagged: 2027 (28.5%)

Sample short reviews:
['good', 'good', 'nice', 'not working properly', 'exilent', 'good', 'very nice', 'ola worst no', 'very nice', "it's useful lot.."]


In [13]:
def detect_non_english(text):
    """
    Flags reviews containing non-Latin script characters.
    Catches: Hindi (Devanagari), Tamil, Kannada, Telugu, Malayalam.
    
    Named is_non_english (not is_hinglish) because detection
    cannot distinguish between Indian language scripts.
    VADER reliability is reduced for all flagged reviews.
    """
    if not isinstance(text, str):
        return False
    
    # Unicode ranges for common Indian scripts
    devanagari = bool(re.search(r'[\u0900-\u097F]', text))  # Hindi
    tamil      = bool(re.search(r'[\u0B80-\u0BFF]', text))  # Tamil
    kannada    = bool(re.search(r'[\u0C80-\u0CFF]', text))  # Kannada
    telugu     = bool(re.search(r'[\u0C00-\u0C7F]', text))  # Telugu
    malayalam  = bool(re.search(r'[\u0D00-\u0D7F]', text))  # Malayalam
    
    return any([devanagari, tamil, kannada, telugu, malayalam])

df['is_non_english'] = df['content'].apply(detect_non_english)

print(f"Non-English reviews flagged: {df['is_non_english'].sum()} ({df['is_non_english'].mean()*100:.1f}%)")
print(f"\nSample non-English reviews:")
print(df[df['is_non_english']]['content'].head(5).tolist())

Non-English reviews flagged: 104 (1.5%)

Sample non-English reviews:
['मस्त', "nowadays it's working well इंप्रूवमेंट हुआ है सुधार हुआ है जिसका असर दिखता है बस थोड़ा सा सर्विस और सुधार ले मेरी गाड़ी 6 महीना सर्विस सेंटर पर खड़ी थी", 'बहुत ही jada kharab सर्विस', 'Ola s1x app and scooty काम नहीं कर रहा है समय बर्बाद हो रहा है ओला सेवा में कोई भी सुनवाई नहीं हो रही है ओला स्कूटी में कई समस्याएं हैं ओला में मेरा अनुभव खराब रहा।ola s1x me कस्टमर केयर एम कोई भी सुनावी नहीं होती है मने कई बार कस्टमर केयर कॉल की है पीआर कोई समस्या हल नहीं हुई है', 'bad service मेरी स्कूटर ola s1 air आपके शोरूम में करीब 6 महीने से खड़ी है जिस पर कोई काम नहीं कर रहे हैं जॉब कार्ड जॉब कार्ड ओपन करने में ही केरीब 3 से 4 महीने लगा दिए अभी 20दिन पहले ही जॉब कार्ड ओपन किया इतनी गंदी सर्विसेज आपकी कि पूछो मत,मत लो रे भैया गाड़ी अगर अपनी जिंदगी को जहन्नुम बनानी है तो इस गाड़ी को लेलो']


In [14]:
print("=" * 55)
print("POST-CLEANING VALIDATION REPORT")
print("=" * 55)

# 1. Shape comparison
print(f"\nShape before cleaning : {original_shape}")
print(f"Shape after cleaning  : {df.shape}")
print(f"Rows retained         : {len(df)} / {original_shape[0]} ({len(df)/original_shape[0]*100:.1f}%)")

# 2. Empty reviews after cleaning
empty_after = (df['content_clean'].str.strip() == '').sum()
print(f"\nEmpty reviews after cleaning: {empty_after}")

# 3. Null check on core columns
core_columns = ['reviewId', 'content', 'content_clean', 'score', 'at',
                'year', 'year_month', 'rating_category', 'word_count']
print(f"\nNull check on core columns:")
print(df[core_columns].isnull().sum())

# 4. Engineered column summary
print(f"\nEngineered columns summary:")
print(f"  developer_replied  : {df['developer_replied'].sum()} True ({df['developer_replied'].mean()*100:.1f}%)")
print(f"  is_short_review    : {df['is_short_review'].sum()} flagged ({df['is_short_review'].mean()*100:.1f}%)")
print(f"  is_non_english     : {df['is_non_english'].sum()} flagged ({df['is_non_english'].mean()*100:.1f}%)")

# 5. Rating category distribution
print(f"\nRating category distribution:")
print(df['rating_category'].value_counts())

print("\n" + "=" * 55)
print("Validation complete. Safe to save.")
print("=" * 55)

POST-CLEANING VALIDATION REPORT

Shape before cleaning : (7119, 12)
Shape after cleaning  : (7119, 22)
Rows retained         : 7119 / 7119 (100.0%)

Empty reviews after cleaning: 0

Null check on core columns:
reviewId           0
content            0
content_clean      0
score              0
at                 0
year               0
year_month         0
rating_category    0
word_count         0
dtype: int64

Engineered columns summary:
  developer_replied  : 0 True (0.0%)
  is_short_review    : 2027 flagged (28.5%)
  is_non_english     : 104 flagged (1.5%)

Rating category distribution:
rating_category
Negative    4689
Positive    2054
Neutral      376
Name: count, dtype: int64

Validation complete. Safe to save.


In [16]:
# Core temporal analysis backbone
# Used in Phase 5 EDA and for Hypothesis 001 and 002 testing

monthly = df.groupby('year_month').agg(
    review_count        = ('reviewId', 'count'),
    avg_rating          = ('score', 'mean'),
    pct_one_star        = ('score', lambda x: (x == 1).mean() * 100),
    pct_five_star       = ('score', lambda x: (x == 5).mean() * 100),
    pct_negative        = ('rating_category', lambda x: (x == 'Negative').mean() * 100),
    avg_review_length   = ('clean_review_length', 'mean'),
    avg_word_count      = ('word_count', 'mean')
).reset_index()

# Round for readability
monthly = monthly.round(2)

# Convert period to string for CSV compatibility
monthly['year_month'] = monthly['year_month'].astype(str)

print(f"Monthly aggregation table created.")
print(f"Shape: {monthly.shape}")
print(f"\nSample (most recent 6 months):")
print(monthly.tail(6).to_string(index=False))

Monthly aggregation table created.
Shape: (50, 8)

Sample (most recent 6 months):
year_month  review_count  avg_rating  pct_one_star  pct_five_star  pct_negative  avg_review_length  avg_word_count
   2026-01           384        2.43         57.29          28.39         61.20              76.05           13.63
   2026-02           315        2.27         60.95          25.40         66.35              79.54           14.35
   2026-03           311        2.60         54.34          34.08         57.88              66.00           12.05
   2026-04           231        2.65         51.52          33.77         56.28              71.53           13.13
   2026-05           232        2.54         55.17          31.03         58.62              91.45           17.17
   2026-06           184        2.87         46.74          39.67         51.09              75.40           13.65


In [17]:
# Convert year_month Period to string for CSV compatibility
df['year_month'] = df['year_month'].astype(str)
df['quarter']    = df['quarter'].astype(str)

# Save cleaned reviews
cleaned_path = r'C:\Users\Naren karthikeyan\OLA Analysis\data\cleaned\ola_reviews_cleaned.csv'
df.to_csv(cleaned_path, index=False)

# Save monthly trends
monthly_path = r'C:\Users\Naren karthikeyan\OLA Analysis\data\cleaned\ola_monthly_trends.csv'
monthly.to_csv(monthly_path, index=False)

print(f"✓ Cleaned reviews saved  : {cleaned_path}")
print(f"  Rows: {len(df)} | Columns: {df.shape[1]}")
print(f"  Size: {os.path.getsize(cleaned_path)/1024:.1f} KB")

print(f"\n✓ Monthly trends saved   : {monthly_path}")
print(f"  Rows: {len(monthly)} | Columns: {monthly.shape[1]}")
print(f"  Size: {os.path.getsize(monthly_path)/1024:.1f} KB")

✓ Cleaned reviews saved  : C:\Users\Naren karthikeyan\OLA Analysis\data\cleaned\ola_reviews_cleaned.csv
  Rows: 7119 | Columns: 22
  Size: 2741.0 KB

✓ Monthly trends saved   : C:\Users\Naren karthikeyan\OLA Analysis\data\cleaned\ola_monthly_trends.csv
  Rows: 50 | Columns: 8
  Size: 2.4 KB


In [19]:
observations_md = """# OLA Electric VoC Analysis — Observations Register

Maintained throughout the project. Every finding labeled per OHC framework:
- **Observation:** Directly supported by data
- **Hypothesis:** Possible explanation requiring further testing  
- **Conclusion:** Validated by statistical evidence (p-value reported)

---

## Phase 1 — Data Collection

| ID | Label | Finding | Evidence |
|----|-------|---------|----------|
| 001 | Observation | 59.9% of 7,119 reviews are 1-star ratings | Rating distribution, combined dataset |
| 002 | Observation | OLA Electric has a 0% developer response rate across 4 years and 7,119 reviews | replyContent column — 100% null |
| 003 | Observation | Review volume spiked 274% between June 2025 (178) and July 2025 (667) | Monthly breakdown, Phase 1 sanity check |
| 004 | Observation | Zero duplicate review IDs across combined dataset of 7,119 reviews | Deduplication check, Phase 1 |
| 005 | Hypothesis | The July 2025 volume spike is linked to a specific identifiable external event | Requires cross-referencing with OLA news timeline in Phase 5 |
| 006 | Hypothesis | Declining review volume in late 2025–2026 reflects disengagement not recovery | Requires sentiment trend analysis in Phase 5 |

---

## Phase 2 — Data Cleaning

| ID | Label | Finding | Evidence |
|----|-------|---------|----------|
| 007 | Observation | {non_english_count} reviews ({non_english_pct:.1f}%) contain non-Latin script — VADER reliability reduced for this subset | is_non_english flag, Phase 2 |
| 008 | Observation | {short_review_count} reviews ({short_review_pct:.1f}%) contain fewer than 4 words | is_short_review flag, Phase 2 |
| 009 | Observation | Average cleaning removed {avg_removed:.1f} characters per review | length_change column, Phase 2 |

---

## Phase 3 — Sentiment Analysis
*To be populated*

## Phase 4 — Complaint Classification
*To be populated*

## Phase 5 — EDA
*To be populated*

## Phase 6 — Statistical Testing
*To be populated*

""".format(
    non_english_count  = df['is_non_english'].sum(),
    non_english_pct    = df['is_non_english'].mean() * 100,
    short_review_count = df['is_short_review'].sum(),
    short_review_pct   = df['is_short_review'].mean() * 100,
    avg_removed        = df['length_change'].mean()
)

# Save
obs_path = r'C:\Users\Naren karthikeyan\OLA Analysis\outputs\observations.md'
with open(obs_path, 'w') as f:
    f.write(observations_md)

print(f"✓ Observations register initialized: {obs_path}")
print(f"\nPhase 2 entries logged:")
print(f"  OBS-007 — Non-English reviews: {df['is_non_english'].sum()} ({df['is_non_english'].mean()*100:.1f}%)")
print(f"  OBS-008 — Short reviews: {df['is_short_review'].sum()} ({df['is_short_review'].mean()*100:.1f}%)")
print(f"  OBS-009 — Avg chars removed: {df['length_change'].mean():.1f}")

✓ Observations register initialized: C:\Users\Naren karthikeyan\OLA Analysis\outputs\observations.md

Phase 2 entries logged:
  OBS-007 — Non-English reviews: 104 (1.5%)
  OBS-008 — Short reviews: 2027 (28.5%)
  OBS-009 — Avg chars removed: 0.8


In [20]:
# Understand the short review distribution better
print("Word count distribution for short reviews:")
print(df[df['is_short_review']]['word_count'].value_counts().sort_index())

print("\nMost common short reviews:")
print(df[df['is_short_review']]['content_clean'].value_counts().head(20))

Word count distribution for short reviews:
word_count
1    858
2    737
3    432
Name: count, dtype: int64

Most common short reviews:
content_clean
good                    358
nice                     77
super                    50
very good                49
ok                       48
good 👍                   35
Good                     28
very nice                24
excellent                21
very bad                 20
good app                 19
best                     19
not working              18
not working properly     17
worst app                16
worst                    15
👍                        15
nice app                 14
superb                   13
bad service              13
Name: count, dtype: int64


In [21]:
# Log updated short review finding to observations register
additional_obs = """
### Phase 2 Addendum — Short Review Analysis

**OBS-012:** 28.5% of reviews (2,027) contain fewer than 4 words.
Content inspection confirms genuine sentiment split:
- Positive camp: 'good' (358), 'nice' (77), 'super' (50), 'excellent' (21)
- Negative camp: 'very bad' (20), 'not working' (18), 'worst' (15), 'bad service' (13)
These reviews carry valid sentiment signal but insufficient text for complaint 
classification. Will be assigned 'Uncategorized — Insufficient Text' in Phase 4.

**OBS-013:** Emoji characters retained in content_clean. VADER has partial emoji 
support. 15 emoji-only reviews will receive near-neutral scores despite positive 
intent. Proportion negligible at aggregate level.

| word_count | review_count | % of short reviews |
|------------|-------------|-------------------|
| 1 word     | 858         | 42.3%             |
| 2 words    | 737         | 36.4%             |
| 3 words    | 432         | 21.3%             |
"""

# Append to existing observations file
with open('../outputs/observations.md', 'a') as f:
    f.write(additional_obs)

print("✓ Observations 012 and 013 logged.")

✓ Observations 012 and 013 logged.
